In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

In [2]:
np.random.seed(1)
categorías=['Guadalajara','Zapopan','Ciudad de México','Monterrey','Morelia','Mexicali']
ponderación=[1/4,1/5,1/4,7/40,1/20,3/40]
limites=[(3500,6000),(3000,12000),(4000,18000),(3000,6000),(2000,5000),(3500,5000)]
ciudad=np.random.choice(categorías,size=20,p=ponderación)
rentas=[]
for c in ciudad:
  rentas.append(np.random.randint(*limites[categorías.index(c)]))
df=pd.DataFrame()
df['Rentas']=rentas
df['Ciudad']=ciudad
df.head()

,Rentas,Ciudad
0,9285,Zapopan
1,5025,Monterrey
2,4531,Guadalajara
3,7415,Zapopan
4,4837,Guadalajara


# Codificación One-Hot

In [3]:
pd.get_dummies(df['Ciudad'])

,Ciudad de México,Guadalajara,Monterrey,Morelia,Zapopan
0,False,False,False,False,True
1,False,False,True,False,False
2,False,True,False,False,False
3,False,False,False,False,True
4,False,True,False,False,False
5,False,True,False,False,False
6,False,True,False,False,False
7,False,False,False,False,True
8,False,False,False,False,True
9,True,False,False,False,False


In [8]:
df_OH=pd.get_dummies(df,prefix='',prefix_sep='')
df_OH.head()

,Rentas,Ciudad de México,Guadalajara,Monterrey,Morelia,Zapopan
0,9285,False,False,False,False,True
1,5025,False,False,True,False,False
2,4531,False,True,False,False,False
3,7415,False,False,False,False,True
4,4837,False,True,False,False,False


In [17]:
(df_OH.iloc[:,1:].sum(axis=1) != 1).any() # La suma de las variables siempre es igual a 1

np.False_

# Codificación Ficticia

In [10]:
df_F=pd.get_dummies(df,drop_first=True,prefix='',prefix_sep='')
df_F.head(10)

,Rentas,Guadalajara,Monterrey,Morelia,Zapopan
0,9285,False,False,False,True
1,5025,False,True,False,False
2,4531,True,False,False,False
3,7415,False,False,False,True
4,4837,True,False,False,False
5,5420,True,False,False,False
6,3816,True,False,False,False
7,11920,False,False,False,True
8,9771,False,False,False,True
9,4431,False,False,False,False


In [18]:
(df_F.iloc[:,1:].sum(axis=1) != 1).any()

np.True_

In [19]:
df['Ciudad'].unique() # Categorías que estoy codificando en la variable 'Ciudad'

array(['Zapopan', 'Monterrey', 'Guadalajara', 'Ciudad de México',
       'Morelia'], dtype=object)

In [23]:
df_F.columns[1:] # Variables que estoy generando

Index(['Guadalajara', 'Monterrey', 'Morelia', 'Zapopan'], dtype='object')

In [28]:
# Categoría de referencia
CR, = set(df['Ciudad'].unique())-set(df_F.columns)
CR

'Ciudad de México'

# Codificación de Efectos

In [31]:
# Paso 1.
df_E=pd.get_dummies(df,drop_first=True,prefix='',prefix_sep='') # Codificación ficticia
CR7, = set(df['Ciudad'].unique())-set(df_E.columns)             # Categoría de referencia 7
df_E[df['Ciudad']==CR7]                                         # Filtro de la Categoría de referencia

,Rentas,Guadalajara,Monterrey,Morelia,Zapopan
9,4431,False,False,False,False
11,16921,False,False,False,False
15,5478,False,False,False,False
17,16803,False,False,False,False


In [37]:
# Paso 2
df_E.iloc[:,1:]=df_E.iloc[:,1:].astype(int)
df_E.head()

,Rentas,Guadalajara,Monterrey,Morelia,Zapopan
0,9285,0,0,0,1
1,5025,0,1,0,0
2,4531,1,0,0,0
3,7415,0,0,0,1
4,4837,1,0,0,0


In [38]:
# Paso 3
df_a = df_E[df['Ciudad']==CR7].astype(int)
df_a

,Rentas,Guadalajara,Monterrey,Morelia,Zapopan
9,4431,0,0,0,0
11,16921,0,0,0,0
15,5478,0,0,0,0
17,16803,0,0,0,0


In [42]:
df_E.iloc[df_a.index,1:]=df_a.iloc[:,1:].replace(0,-1)
df_E.head(10)

,Rentas,Guadalajara,Monterrey,Morelia,Zapopan
0,9285,0,0,0,1
1,5025,0,1,0,0
2,4531,1,0,0,0
3,7415,0,0,0,1
4,4837,1,0,0,0
5,5420,1,0,0,0
6,3816,1,0,0,0
7,11920,0,0,0,1
8,9771,0,0,0,1
9,-1,-1,-1,-1,-1


#Regresión lineal